In [0]:
from pyspark.sql.functions import *

In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"

In [0]:
df = spark.table('customer360_cata.bronze.customers')

In [0]:
df = df.withColumn(
    "dob",
    coalesce(
        try_to_date(col("dob"), "yyyy/MM/dd"),
        try_to_date(col("dob"), "dd-MM-yyyy"),
        try_to_date(col("dob"), "yyyy-MM-dd")
    )
)

In [0]:
df =df.drop("_rescued_data")

In [0]:
location_df = spark.table('customer360_cata.bronze.locations')

In [0]:
df = df.join(location_df,df.location_id == location_df.location_id,'inner').select("customer_id","name","email","gender","dob","city","state","country","zipcode")


In [0]:
df = df.withColumn('email',lower(col("email")))\
        .withColumn("name",initcap(col("name")))

In [0]:
df = df.withColumn("gender",initcap(when(col('gender').isin(["F","f"]) ,"Female")\
    .when(col('gender').isin(["M","m"]) ,"Male")\
        .when(col('gender').isNull(),"Unknown")\
        .otherwise(col('gender'))))

In [0]:
df = df.dropDuplicates(["customer_id"])

In [0]:
df =df.dropna(subset=["customer_id","email","name"])

In [0]:
df.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", "true") \
    .option('path',base_adls2_path + '/silver/customers') \
        .saveAsTable('customer360_cata.silver.customers')

In [0]:
%sql
--select * from customer360_cata.silver.customers

In [0]:
%sql
---drop table customer360_cata.silver.customers